In [9]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [3]:
# data: https://www.kaggle.com/datasets/vipullrathod/fish-market
data = pd.read_csv('../datasets/Fish.csv')
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 159 entries, 0 to 158
Data columns (total 7 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   Species  159 non-null    object 
 1   Weight   159 non-null    float64
 2   Length1  159 non-null    float64
 3   Length2  159 non-null    float64
 4   Length3  159 non-null    float64
 5   Height   159 non-null    float64
 6   Width    159 non-null    float64
dtypes: float64(6), object(1)
memory usage: 8.8+ KB


In [4]:
data.head()

,Species,Weight,Length1,Length2,Length3,Height,Width
0,Bream,242.0,23.2,25.4,30.0,11.5200,4.0200
1,Bream,290.0,24.0,26.3,31.2,12.4800,4.3056
2,Bream,340.0,23.9,26.5,31.1,12.3778,4.6961
3,Bream,363.0,26.3,29.0,33.5,12.7300,4.4555
4,Bream,430.0,26.5,29.0,34.0,12.4440,5.1340


In [12]:
# feature engineering
categorical_features = ['Species']
numerical_features = ['Length1', 'Length2', 'Length3', 'Height', 'Width']

X = data.drop('Weight', axis=1)
y = data['Weight']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

# pipeline
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

# split data (FIRST!)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# model
model = pipeline.fit(X_train, y_train)

# Predictions
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

# feature_importance
preprocessor = model.named_steps['preprocessor']
encoder = preprocessor.named_transformers_['cat']
categorical_features_ = encoder.get_feature_names_out(['Species'])

features = list(numerical_features) + list(categorical_features_)
coefficients = model.named_steps['regressor'].coef_

feature_importance = pd.DataFrame({
    'feature': features,
    'coefficient': coefficients,
    'abs_coefficient': np.abs(coefficients)
}).sort_values('abs_coefficient', ascending=False)

print("="*50)
print("FEATURE IMPORTANCE")
print("="*50)
print(feature_importance)

FEATURE IMPORTANCE
              feature  coefficient  abs_coefficient
1             Length2   590.247377       590.247377
0             Length1  -587.214367       587.214367
2             Length3   446.205234       446.205234
8        Species_Pike  -373.686995       373.686995
10      Species_Smelt   288.374964       288.374964
6      Species_Parkki    97.250915        97.250915
3              Height   -42.740526        42.740526
5       Species_Bream   -42.311511        42.311511
7       Species_Perch    32.359289        32.359289
4               Width    11.765117        11.765117
9       Species_Roach    -7.316676         7.316676
11  Species_Whitefish     5.330013         5.330013


In [13]:
# performance metrics
r2_train = r2_score(y_train, y_train_pred)
r2_test = r2_score(y_test, y_test_pred)
mse_train = mean_squared_error(y_train, y_train_pred)
mse_test = mean_squared_error(y_test, y_test_pred)
rmse_train = np.sqrt(mse_train)
rmse_test = np.sqrt(mse_test)
mae_train = mean_absolute_error(y_train, y_train_pred)
mae_test = mean_absolute_error(y_test, y_test_pred)

print("\n" + "="*50)
print(f"Model Performance:")
print("="*50)

print(f"\n📊 R²")
print(f"   • Model explains {r2_test*100:.1f}% of the variation in house price of unit area")
print(f"   • Only {100-r2_test*100:.1f}% of variation is unexplained (due to other factors)")
print(f"Training R²: {r2_train:.4f}")
print(f"Test R²: {r2_test:.4f}")

if abs(r2_train - r2_test) < 0.05:
    print("✅ Good: Training and testing R² are similar - no overfitting")
else:
    print(f"⚠️  Warning: Difference of {abs(r2_train - r2_test):.3f} between training and testing R²")

print()

# print(f"\n📊 Cross Validation:")
# cv_scores = cross_val_score(model, X, y, cv=5)
# print(f"Cross-validation R²: {cv_scores.mean():.3f} (+/- {cv_scores.std():.3f})")

# print()

print(f"\n📊 MSE:")
print(f"Training MSE: {mse_train:.2f}")
print(f"Test MSE: {mse_test:.2f}")

print()

print(f"\n📊 RMSE")
print(f"   • Predictions are off by ±{rmse_test:.2f} on average")
print(f"   • In other words, 68% of predictions fall within {rmse_test:.2f} of actual value")
print(f"   • 95% of predictions fall within {rmse_test*2:.2f} of actual value")
print(f"Training RMSE: {rmse_train:.2f}")
print(f"Testing RMSE: {rmse_test:.2f}")

print()

print(f"\n📊 MAE:")
print(f"Training MAE: {mae_train:.2f}")
print(f"Testing MAE: {mae_test:.2f}")


Model Performance:

📊 R²
   • Model explains 95.1% of the variation in house price of unit area
   • Only 4.9% of variation is unexplained (due to other factors)
Training R²: 0.9286
Test R²: 0.9507
✅ Good: Training and testing R² are similar - no overfitting


📊 MSE:
Training MSE: 8777.60
Test MSE: 7007.38


📊 RMSE
   • Predictions are off by ±83.71 on average
   • In other words, 68% of predictions fall within 83.71 of actual value
   • 95% of predictions fall within 167.42 of actual value
Training RMSE: 93.69
Testing RMSE: 83.71


📊 MAE:
Training MAE: 69.17
Testing MAE: 65.30
